In [ ]:
install.packages(c(
  "tidyverse",
  "sf",
  "lubridate",
  "leaflet",
  "ggplot2",
  "janitor",
  "readr",
  "stringr"
))
library(tidyverse)
library(sf)
library(lubridate)
library(leaflet)
library(janitor)
library(stringr)

gtfs_path <- "C:/Users/beenj/OneDrive/Escritorio/Proyecto Data Science/gtfs-valparaiso"
list.files(gtfs_path)
agency <- read_csv(file.path(gtfs_path, "agency.txt"))
routes <- read_csv(file.path(gtfs_path, "routes.txt"))
trips <- read_csv(file.path(gtfs_path, "trips.txt"))
stops <- read_csv(file.path(gtfs_path, "stops.txt"))
stop_times <- read_csv(file.path(gtfs_path, "stop_times.txt"))
calendar <- read_csv(file.path(gtfs_path, "calendar.txt"))
shapes <- read_csv(file.path(gtfs_path, "shapes.txt"))

resumen_gtfs <- tibble(
  archivo = c("agency", "routes", "trips", "stops", "stop_times", "calendar", "shapes"),
  filas = c(
    nrow(agency),
    nrow(routes),
    nrow(trips),
    nrow(stops),
    nrow(stop_times),
    nrow(calendar),
    nrow(shapes)
  )
)

resumen_gtfs

n_rutas <- n_distinct(routes$route_id)
n_viajes <- n_distinct(trips$trip_id)
n_paraderos <- n_distinct(stops$stop_id)
n_operadores <- n_distinct(agency$agency_id)

tibble(
  operadores = n_operadores,
  rutas = n_rutas,
  viajes = n_viajes,
  paraderos = n_paraderos
)

routes %>%
  select(route_id, route_short_name, route_long_name, route_type) %>%
  head(20)

rutas_corredor <- routes %>%
  filter(str_detect(
    str_to_lower(route_long_name),
    "placilla|curauma|baron|barón|peñuelas|penuelas"
  )) %>%
  select(route_id, route_short_name, route_long_name, agency_id, route_type)

rutas_corredor

rutas_excluidas <- c("001", "002")

# Paraderos zona Placilla / Curauma / Peñuelas
paraderos_placilla <- stops %>%
  filter(str_detect(
    str_to_lower(stop_name),
    "placilla|curauma|peñuelas|penuelas"
  )) %>%
  select(stop_id, stop_name, stop_lat, stop_lon)

# Paraderos zona Valparaíso / centro / Barón
paraderos_valpo <- stops %>%
  filter(str_detect(
    str_to_lower(stop_name),
    "baron|barón|aduana|argentina|santos ossa|rodoviario|congreso|puerto|victoria|pedro montt|chacabuco"
  )) %>%
  select(stop_id, stop_name, stop_lat, stop_lon)

# Viajes que pasan por Placilla
viajes_placilla <- stop_times %>%
  semi_join(paraderos_placilla, by = "stop_id") %>%
  distinct(trip_id)

# Viajes que pasan por Valparaíso
viajes_valpo <- stop_times %>%
  semi_join(paraderos_valpo, by = "stop_id") %>%
  distinct(trip_id)

# Viajes que pasan por ambas zonas
viajes_corredor_correcto <- viajes_placilla %>%
  inner_join(viajes_valpo, by = "trip_id")

# Rutas asociadas a esos viajes
rutas_corredor_correcto <- trips %>%
  semi_join(viajes_corredor_correcto, by = "trip_id") %>%
  left_join(routes, by = "route_id") %>%
  count(
    route_id,
    route_short_name,
    route_long_name,
    direction_id,
    trip_headsign,
    name = "n_viajes"
  ) %>%
  arrange(route_short_name, direction_id, desc(n_viajes))

rutas_corredor_actual <- rutas_corredor_correcto %>%
  filter(!route_short_name %in% rutas_excluidas)

rutas_corredor_actual

rutas_corredor_actual <- rutas_corredor_correcto %>%
  filter(
    str_detect(
      str_to_lower(paste(route_short_name, route_long_name, trip_headsign)),
      "placilla|invica|universidad"
    )
  ) %>%
  filter(
    !route_short_name %in% c("001", "002")
  )

rutas_corredor_actual
rutas_placilla_valpo <- rutas_corredor_actual %>%
  filter(route_short_name %in% c("901", "902", "901 Inyector Nocturno"))

rutas_placilla_valpo

rutas_placilla_valpo <- rutas_corredor_actual %>%
  filter(route_short_name %in% c(
    "901",
    "902",
    "903",
    "901 Inyector Nocturno"
  ))

rutas_placilla_valpo

routes %>%
  filter(str_detect(
    str_to_lower(route_long_name),
    "trole|eléctrico|electrico"
  )) %>%
  select(route_id, route_short_name, route_long_name, agency_id, route_type)
agency
rutas_trole <- routes %>%
  left_join(agency, by = "agency_id") %>%
  filter(str_detect(
    str_to_lower(agency_name),
    "trole"
  ))

rutas_trole

rutas_trole_electricos <- routes %>%
  filter(str_detect(
    str_to_lower(route_short_name),
    "801|802|e01|e02|e03|e04|e05"
  ) | str_detect(
    str_to_lower(route_long_name),
    "trole|electrico|eléctrico"
  ))

rutas_trole_electricos
rutas_base_modelo <- bind_rows(
  rutas_corredor_actual,
  rutas_trole_electricos
) %>%
  distinct(route_id, .keep_all = TRUE)

paraderos_modelo <- stop_times %>%
  count(stop_id, name = "frecuencia") %>%
  left_join(stops, by = "stop_id") %>%
  arrange(desc(frecuencia))

rutas_placilla_valpo <- rutas_corredor_actual %>%
  filter(route_short_name %in% c(
    "901",
    "902",
    "903",
    "901 Inyector Nocturno"
  ))

route_ids_placilla_valpo <- rutas_placilla_valpo %>%
  pull(route_id) %>%
  unique()

trips_placilla_valpo <- trips %>%
  filter(route_id %in% route_ids_placilla_valpo)

stop_times_placilla_valpo <- stop_times %>%
  semi_join(trips_placilla_valpo, by = "trip_id")

exists("stop_times_placilla_valpo")
nrow(stop_times_placilla_valpo)

paraderos_modelo <- stop_times %>%
  count(stop_id, name = "frecuencia") %>%
  left_join(stops, by = "stop_id") %>%
  arrange(desc(frecuencia))

paraderos_modelo <- stop_times %>%
  count(stop_id, name = "frecuencia_programada") %>%
  left_join(stops, by = "stop_id") %>%
  arrange(desc(frecuencia_programada))

head(paraderos_modelo, 20)

library(sf)

paraderos_modelo_sf <- paraderos_modelo %>%
  st_as_sf(coords = c("stop_lon", "stop_lat"), crs = 4326)

library(ggplot2)

ggplot() +
  geom_sf(data = paraderos_modelo_sf, aes(size = frecuencia_programada), alpha = 0.6) +
  labs(
    title = "Paraderos del eje Placilla–Valparaíso",
    subtitle = "Tamaño según frecuencia programada",
    size = "Frecuencia"
  )
library(leaflet)

leaflet(paraderos_modelo_sf) %>%
  addTiles() %>%
  addCircleMarkers(
    radius = ~sqrt(frecuencia_programada) / 2,
    popup = ~paste0(
      "<b>", stop_name, "</b><br>",
      "Frecuencia programada: ", frecuencia_programada
    )
  )
top_paraderos <- paraderos_modelo %>%
  select(stop_id, stop_name, frecuencia_programada, stop_lat, stop_lon) %>%
  arrange(desc(frecuencia_programada)) %>%
  slice_head(n = 20)

top_paraderos

write_csv(paraderos_modelo, "paraderos_eje_placilla_valparaiso.csv")
write_csv(top_paraderos, "top_20_paraderos_eje.csv")
library(tidyverse)
library(stringr)

# Función para convertir hora a segundos
time_to_seconds <- function(x) {
  parts <- str_split(x, ":", simplify = TRUE)
  as.numeric(parts[,1]) * 3600 +
    as.numeric(parts[,2]) * 60 +
    as.numeric(parts[,3])
}

# Limpiar tiempos
stop_times_limpio <- stop_times %>%
  filter(!is.na(arrival_time)) %>%
  mutate(
    arrival_sec = time_to_seconds(arrival_time)
  )

# Calcular duración por viaje
duracion_viajes <- stop_times_limpio %>%
  group_by(trip_id) %>%
  summarise(
    inicio = min(arrival_sec),
    fin = max(arrival_sec),
    duracion_min = (fin - inicio) / 60,
    .groups = "drop"
  )
ggplot(duracion_viajes, aes(x = duracion_min)) +
  geom_histogram(bins = 60, fill = "steelblue", alpha = 0.8) +
  labs(
    title = "Distribución de duración de viajes en el sistema",
    x = "Duración del viaje (minutos)",
    y = "Cantidad de viajes"
  )
# Filtrar duraciones realistas
duracion_viajes_filtrado <- duracion_viajes %>%
  filter(duracion_min >= 5, duracion_min <= 180)

# Calcular media y mediana
media_duracion <- mean(duracion_viajes_filtrado$duracion_min, na.rm = TRUE)
mediana_duracion <- median(duracion_viajes_filtrado$duracion_min, na.rm = TRUE)

# Gráfico mejorado
ggplot(duracion_viajes_filtrado, aes(x = duracion_min)) +
  geom_histogram(
    bins = 50,
    fill = "#2C7FB8",
    color = "white",
    alpha = 0.85
  ) +
  geom_vline(
    xintercept = media_duracion,
    color = "#D7191C",
    linewidth = 1,
    linetype = "dashed"
  ) +
  geom_vline(
    xintercept = mediana_duracion,
    color = "#FDAE61",
    linewidth = 1,
    linetype = "dashed"
  ) +
  annotate(
    "text",
    x = media_duracion + 8,
    y = Inf,
    label = paste0("Media: ", round(media_duracion, 1), " min"),
    vjust = 2,
    color = "#D7191C",
    size = 4
  ) +
  annotate(
    "text",
    x = mediana_duracion + 8,
    y = Inf,
    label = paste0("Mediana: ", round(mediana_duracion, 1), " min"),
    vjust = 4,
    color = "#FDAE61",
    size = 4
  ) +
  labs(
    title = "Distribución de duración de viajes programados",
    subtitle = "Sistema de transporte público del Gran Valparaíso",
    x = "Duración del viaje (minutos)",
    y = "Cantidad de viajes",
    caption = "Fuente: Elaboración propia a partir de feed GTFS del Gran Valparaíso"
  ) +
  theme_minimal()

library(dplyr)
library(ggplot2)

# --- Proxy de demanda con ajuste poblacional ---
demand_2012 <- 10000
pop_2012 <- 1700000
pop_2024 <- 1896053
factor_growth <- pop_2024 / pop_2012
demand_2024 <- demand_2012 * factor_growth

# Proxy por frecuencia usando paraderos_modelo
proxy_demanda <- paraderos_modelo %>%
  mutate(weight = frecuencia_programada / sum(frecuencia_programada)) %>%
  mutate(demand_proxy = round(weight * demand_2024)) %>%
  select(stop_name, demand_proxy)

# --- Baseline gravitacional calibrado ---
demand_proxy_total <- 11150  # viajes/día

estaciones <- data.frame(
  estacion = c("Placilla", "Intermedia", "Barón"),
  poblacion_origen = c(20000, 15000, 25000),
  atractivo_destino = c(300000, 300000, 300000),
  distancia_min = c(50, 40, 30)
)

alpha <- 1.5

estaciones <- estaciones %>%
  mutate(
    demanda_grav_raw = (poblacion_origen * atractivo_destino) / (distancia_min ^ alpha)
  )

factor_k <- demand_proxy_total / sum(estaciones$demanda_grav_raw)

estaciones <- estaciones %>%
  mutate(
    demanda_grav_calibrada = round(demanda_grav_raw * factor_k)
  )

# --- Comparación en gráfico de barras ---
proxy_plot <- proxy_demanda %>%
  mutate(modelo = "Proxy Frecuencia")

grav_plot <- estaciones %>%
  select(estacion, demanda_grav_calibrada) %>%
  rename(stop_name = estacion, demand_proxy = demanda_grav_calibrada) %>%
  mutate(modelo = "Gravitacional")

comparacion <- bind_rows(proxy_plot, grav_plot)

ggplot(comparacion, aes(x = stop_name, y = demand_proxy, fill = modelo)) +
  geom_col(position = "dodge") +
  labs(
    title = "Comparación de demanda estimada",
    subtitle = "Proxy por frecuencia vs. modelo gravitacional calibrado",
    x = "Estación / Paradero",
    y = "Viajes/día"
  ) +
  theme_minimal()

# --- Scatter plot comparativo ---
comparacion_scatter <- proxy_plot %>%
  inner_join(grav_plot, by = "stop_name", suffix = c("_proxy", "_grav"))

ggplot(comparacion_scatter, aes(x = demand_proxy_proxy, y = demand_proxy_grav, label = stop_name)) +
  geom_point(color = "steelblue", size = 3, alpha = 0.7) +
  geom_text(vjust = -1, size = 3) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
  labs(
    title = "Comparación Proxy vs. Gravitacional",
    subtitle = "Cada punto representa una estación/paradero",
    x = "Demanda Proxy (frecuencia)",
    y = "Demanda Gravitacional (calibrada)"
  ) +
  theme_minimal()
estaciones <- data.frame(
  stop_name = c("Placilla Oriente", "Intermedia", "Barón"),
  poblacion_origen = c(20000, 15000, 25000),
  atractivo_destino = c(300000, 300000, 300000),
  distancia_min = c(50, 40, 30)
)
library(dplyr)
library(ggplot2)

# --- Proxy de demanda con ajuste poblacional ---
demand_2012 <- 10000
pop_2012 <- 1700000
pop_2024 <- 1896053
factor_growth <- pop_2024 / pop_2012
demand_2024 <- demand_2012 * factor_growth

# Proxy por frecuencia usando paraderos_modelo
proxy_demanda <- paraderos_modelo %>%
  mutate(weight = frecuencia_programada / sum(frecuencia_programada)) %>%
  mutate(demand_proxy = round(weight * demand_2024)) %>%
  select(stop_name, demand_proxy)

# --- Baseline gravitacional calibrado ---
demand_proxy_total <- 11150  # viajes/día

estaciones <- data.frame(
  stop_name = c("Placilla", "Intermedia", "Barón"),
  poblacion_origen = c(20000, 15000, 25000),
  atractivo_destino = c(300000, 300000, 300000),
  distancia_min = c(50, 40, 30)
)

alpha <- 1.5

estaciones <- estaciones %>%
  mutate(
    demanda_grav_raw = (poblacion_origen * atractivo_destino) / (distancia_min ^ alpha)
  )

factor_k <- demand_proxy_total / sum(estaciones$demanda_grav_raw)

estaciones <- estaciones %>%
  mutate(
    demanda_grav_calibrada = round(demanda_grav_raw * factor_k)
  )

# --- Comparación en gráfico de barras ---
proxy_plot <- proxy_demanda %>%
  mutate(modelo = "Proxy Frecuencia")

grav_plot <- estaciones %>%
  select(stop_name, demanda_grav_calibrada) %>%
  rename(demand_proxy = demanda_grav_calibrada) %>%
  mutate(modelo = "Gravitacional")

comparacion <- bind_rows(proxy_plot, grav_plot)

ggplot(comparacion, aes(x = stop_name, y = demand_proxy, fill = modelo)) +
  geom_col(position = "dodge") +
  labs(
    title = "Comparación de demanda estimada",
    subtitle = "Proxy por frecuencia vs. modelo gravitacional calibrado",
    x = "Estación / Paradero",
    y = "Viajes/día"
  ) +
  theme_minimal()

library(dplyr)
library(ggplot2)

# --- Proxy de demanda ---
proxy_demanda <- paraderos_modelo %>%
  mutate(weight = frecuencia_programada / sum(frecuencia_programada)) %>%
  mutate(demand_proxy = round(weight * 11150)) %>%
  # Para simplificar, nos quedamos solo con tres estaciones clave
  filter(str_detect(stop_name, "Placilla|Intermedia|Barón")) %>%
  select(stop_name, demand_proxy)

# --- Baseline gravitacional calibrado ---
estaciones <- data.frame(
  stop_name = c("Placilla", "Intermedia", "Barón"),
  poblacion_origen = c(20000, 15000, 25000),
  atractivo_destino = c(300000, 300000, 300000),
  distancia_min = c(50, 40, 30)
)

alpha <- 1.5
estaciones <- estaciones %>%
  mutate(demanda_grav_raw = (poblacion_origen * atractivo_destino) / (distancia_min ^ alpha))

factor_k <- 11150 / sum(estaciones$demanda_grav_raw)

estaciones <- estaciones %>%
  mutate(demanda_grav_calibrada = round(demanda_grav_raw * factor_k))

# --- Comparación scatter ---
comparacion_scatter <- proxy_demanda %>%
  inner_join(estaciones, by = "stop_name")

ggplot(comparacion_scatter, aes(x = demand_proxy, y = demanda_grav_calibrada, label = stop_name)) +
  geom_point(color = "steelblue", size = 3, alpha = 0.7) +
  geom_text(vjust = -1, size = 3) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
  labs(
    title = "Comparación Proxy vs. Gravitacional",
    subtitle = "Cada punto representa una estación/paradero",
    x = "Demanda Proxy (frecuencia)",
    y = "Demanda Gravitacional (calibrada)"
  ) +
  theme_minimal()

unique(proxy_demanda$stop_name)
unique(estaciones$stop_name)
library(dplyr)
library(ggplot2)
library(stringr)

# Diccionario de correspondencia
correspondencia <- data.frame(
  stop_name = c("Pasarela Placilla De Peñuelas / Poniente",
                "Pasarela Placilla De Peñuelas / Oriente",
                "Placilla - Bernardo Vera",
                "Paradero 7 Placilla",
                "Edificio Torre Barón 2"),
  estacion = c("Placilla", "Placilla", "Placilla", "Placilla", "Barón")
)

# Proxy con correspondencia
proxy_plot <- proxy_demanda %>%
  inner_join(correspondencia, by = "stop_name")

# Gravitacional
estaciones <- estaciones %>%
  mutate(estacion = stop_name)  # asegurar columna 'estacion'

# Unión
comparacion_scatter <- proxy_plot %>%
  inner_join(estaciones %>% select(estacion, demanda_grav_calibrada),
             by = "estacion")

# Scatter plot
ggplot(comparacion_scatter, aes(x = demand_proxy, y = demanda_grav_calibrada, label = stop_name)) +
  geom_point(color = "steelblue", size = 3, alpha = 0.7) +
  geom_text(vjust = -1, size = 3) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
  labs(
    title = "Comparación Proxy vs. Gravitacional",
    subtitle = "Cada punto representa una estación/paradero",
    x = "Demanda Proxy (frecuencia)",
    y = "Demanda Gravitacional (calibrada)"
  ) +
  theme_minimal()
ggplot(comparacion_scatter, aes(x = demand_proxy, y = demanda_grav_calibrada, label = estacion)) +
  geom_point(color = "steelblue", size = 3, alpha = 0.7) +
  geom_text(vjust = -1, size = 3) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
  labs(
    title = "Comparación Proxy vs. Gravitacional",
    subtitle = "Cada punto representa una estación/paradero",
    x = "Demanda Proxy (frecuencia)",
    y = "Demanda Gravitacional (calibrada)"
  ) +
  theme_minimal()
ggplot(comparacion, aes(x = stop_name, y = demand_proxy, fill = modelo)) +
  geom_col(position = "dodge") +
  labs(
    title = "Demanda estimada por estación",
    subtitle = "Comparación Proxy vs. Gravitacional",
    x = "Estación / Paradero",
    y = "Viajes/día"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))


library(dplyr)
library(ggplot2)
library(stringr)

# Agrupar proxy por estación
proxy_estaciones <- proxy_demanda %>%
  mutate(estacion = case_when(
    str_detect(stop_name, "Placilla") ~ "Placilla",
    str_detect(stop_name, "Barón") ~ "Barón",
    TRUE ~ "Intermedia"
  )) %>%
  group_by(estacion) %>%
  summarise(demand_proxy = sum(demand_proxy))

# Gravitacional ya está por estación
grav_estaciones <- estaciones %>%
  select(stop_name, demanda_grav_calibrada) %>%
  rename(estacion = stop_name, demand_grav = demanda_grav_calibrada)

# Unir ambos
comparacion <- proxy_estaciones %>%
  inner_join(grav_estaciones, by = "estacion") %>%
  tidyr::pivot_longer(cols = c(demand_proxy, demand_grav),
                      names_to = "modelo", values_to = "viajes")

# Gráfico de barras comparativo
ggplot(comparacion, aes(x = estacion, y = viajes, fill = modelo)) +
  geom_col(position = "dodge") +
  labs(
    title = "Demanda estimada por estación",
    subtitle = "Proxy por frecuencia vs. Gravitacional calibrado",
    x = "Estación",
    y = "Viajes/día"
  ) +
  theme_minimal()
library(dplyr)
library(ggplot2)
library(tidyr)

# Proxy agrupado por estación
proxy_estaciones <- proxy_demanda %>%
  mutate(estacion = case_when(
    str_detect(stop_name, "Placilla") ~ "Placilla",
    str_detect(stop_name, "Barón") ~ "Barón",
    TRUE ~ "Intermedia"
  )) %>%
  group_by(estacion) %>%
  summarise(demand_proxy = sum(demand_proxy))

# Gravitacional por estación
grav_estaciones <- estaciones %>%
  select(stop_name, demanda_grav_calibrada) %>%
  rename(estacion = stop_name, demand_grav = demanda_grav_calibrada)

# Unir ambos y pasar a formato largo
comparacion <- proxy_estaciones %>%
  inner_join(grav_estaciones, by = "estacion")

comparacion_long <- comparacion %>%
  pivot_longer(cols = c(demand_proxy, demand_grav),
               names_to = "modelo", values_to = "viajes")
ggplot(comparacion_long, aes(x = estacion, y = viajes, fill = modelo)) +
  geom_col() +
  facet_wrap(~modelo, ncol = 1, scales = "fixed") +
  labs(
    title = "Demanda estimada por estación",
    subtitle = "Visualización separada por modelo",
    x = "Estación",
    y = "Viajes/día"
  ) +
  theme_minimal()
# Proxy
ggplot(filter(comparacion_long, modelo == "demand_proxy"),
       aes(x = estacion, y = viajes, fill = estacion)) +
  geom_col() +
  labs(title = "Demanda Proxy por estación",
       x = "Estación", y = "Viajes/día") +
  theme_minimal()

# Gravitacional
ggplot(filter(comparacion_long, modelo == "demand_grav"),
       aes(x = estacion, y = viajes, fill = estacion)) +
  geom_col() +
  labs(title = "Demanda Gravitacional por estación",
       x = "Estación", y = "Viajes/día") +
  theme_minimal()
install.packages("patchwork")

library(patchwork)

grafico_proxy <- ggplot(filter(comparacion_long, modelo == "demand_proxy"),
                        aes(x = estacion, y = viajes, fill = estacion)) +
  geom_col() +
  labs(title = "Demanda Proxy por estación",
       x = "Estación", y = "Viajes/día") +
  theme_minimal() +
  ylim(0, max(comparacion_long$viajes))

grafico_grav <- ggplot(filter(comparacion_long, modelo == "demand_grav"),
                       aes(x = estacion, y = viajes, fill = estacion)) +
  geom_col() +
  labs(title = "Demanda Gravitacional por estación",
       x = "Estación", y = "Viajes/día") +
  theme_minimal() +
  ylim(0, max(comparacion_long$viajes))

# Mostrar juntos
grafico_proxy + grafico_grav


# Proxy
ggplot(filter(comparacion_long, modelo == "demand_proxy"),
       aes(x = estacion, y = viajes, fill = estacion)) +
  geom_col() +
  labs(title = "Demanda Proxy por estación",
       x = "Estación", y = "Viajes/día") +
  theme_minimal() +
  ylim(0, max(comparacion_long$viajes))

# Gravitacional
ggplot(filter(comparacion_long, modelo == "demand_grav"),
       aes(x = estacion, y = viajes, fill = estacion)) +
  geom_col() +
  labs(title = "Demanda Gravitacional por estación",
       x = "Estación", y = "Viajes/día") +
  theme_minimal() +
  ylim(0, max(comparacion_long$viajes))

comparacion_pct <- comparacion_long %>%
  group_by(modelo) %>%
  mutate(pct = viajes / sum(viajes) * 100)

ggplot(comparacion_pct, aes(x = estacion, y = pct, fill = modelo)) +
  geom_col(position = "dodge") +
  labs(
    title = "Distribución porcentual de demanda por estación",
    subtitle = "Proxy vs. Gravitacional",
    x = "Estación",
    y = "% de viajes"
  ) +
  theme_minimal()
library(tidyverse)

# Dataset de sensibilidad
sensibilidad_alpha <- tibble(
  alpha = c(0.8, 0.8, 1.2, 1.2, 1.8, 1.8),
  estacion = c("Barón", "Placilla", "Barón", "Placilla", "Barón", "Placilla"),
  porcentaje_demanda = c(80, 20, 73, 27, 60, 40)
)

# Gráfico
ggplot(sensibilidad_alpha,
       aes(x = factor(alpha),
           y = porcentaje_demanda,
           fill = estacion)) +
  geom_col(position = "dodge") +
  labs(
    title = "Análisis de sensibilidad del modelo gravitacional",
    subtitle = "Distribución de demanda según distintos valores de fricción espacial",
    x = "Parámetro de fricción (α)",
    y = "% de demanda",
    fill = "Estación"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(size = 14, face = "bold"),
    plot.subtitle = element_text(size = 10, margin = margin(b = 15)),
    axis.title = element_text(size = 11),
    legend.position = "right"
  )

# Guardar imagen
ggsave(
  "sensibilidad_alpha.png",
  width = 8,
  height = 5,
  dpi = 300
)